# RQ4 — Robustness to Noisy / Missing Sensor Data
*How robust are predictions when sensor features are corrupted with noise, and which features cause the largest performance drop?*

**Outputs:** `RQ4_robustness.csv`, `RQ4_robustness.pdf`

In [10]:

# ============================================================
# Shared data loading & preprocessing (Algerian Forest Fires)
# ============================================================
import os, glob, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300,
    "font.size": 11, "axes.titlesize": 13, "axes.labelsize": 11,
    "axes.grid": True, "grid.alpha": 0.3, "figure.autolayout": True,
})

OUTDIR = "/kaggle/working"          # on Kaggle this is the output folder
os.makedirs(OUTDIR, exist_ok=True)

def find_csv():
    """Find the Algerian Forest Fires csv anywhere under /kaggle/input."""
    cands = glob.glob("/kaggle/input/**/*.csv", recursive=True)
    if not cands:                   # local fallback
        cands = glob.glob("**/*.csv", recursive=True)
    # prefer a file whose name mentions 'algerian' or 'forest'
    for c in cands:
        n = os.path.basename(c).lower()
        if "algerian" in n or "forest" in n or "fire" in n:
            return c
    return cands[0]

def load_algerian():
    path = find_csv()
    print("Loading:", path)
    with open(path, "r", encoding="utf-8", errors="ignore") as fh:
        lines = fh.read().splitlines()
    # The raw UCI file mixes a title line, two region headers and a blank row,
    # so we parse it line by line rather than with a fixed-width csv reader.
    rows = [ln.split(",") for ln in lines]
    header = None
    region = "Bejaia"           # first block in the UCI file
    region_switched = False
    records, cols = [], None
    for r in rows:
        cells = [str(x).strip() for x in r]
        joined = " ".join(cells).lower()
        if "temperature" in joined and ("rh" in joined or "ws" in joined):
            cols = [c.strip() for c in cells if c.strip() != ""]
            header = cols
            if records and not region_switched:
                region = "Sidi-Bel Abbes"   # second header => second region
                region_switched = True
            continue
        if all(c == "" for c in cells):
            continue
        if "region" in joined or "dataset" in joined:   # title / region label line
            if "sidi" in joined:
                region = "Sidi-Bel Abbes"; region_switched = True
            continue
        if header is None:
            continue
        data_cells = [c for c in cells if c != ""]
        if len(data_cells) < len(cols):
            continue
        rec = dict(zip(cols, data_cells[:len(cols)]))
        rec["region"] = region
        records.append(rec)
    df = pd.DataFrame.from_records(records)
    return df

df = load_algerian()
df.columns = [c.strip().replace(" ", "_") for c in df.columns]

# Standardise the target column name
target_col = [c for c in df.columns if "class" in c.lower()]
target_col = target_col[0] if target_col else df.columns[-2]
df = df.rename(columns={target_col: "Classes"})

# Clean target -> binary (fire = 1, not fire = 0)
df["Classes"] = (df["Classes"].astype(str).str.strip().str.lower()
                 .str.replace(r"\s+", " ", regex=True))
df = df[df["Classes"].isin(["fire", "not fire"])].copy()
df["target"] = (df["Classes"] == "fire").astype(int)

# Numeric feature columns
FEATURES = ["Temperature", "RH", "Ws", "Rain",
            "FFMC", "DMC", "DC", "ISI", "BUI", "FWI"]
FEATURES = [f for f in FEATURES if f in df.columns]
for c in FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=FEATURES + ["target"]).reset_index(drop=True)

print("Shape:", df.shape, "| Fire:", int(df.target.sum()),
      "| Not fire:", int((1-df.target).sum()))
print("Regions:", df["region"].value_counts().to_dict())
X = df[FEATURES].copy()
y = df["target"].copy()


Loading: /kaggle/input/notebooks/sudhanshu432/eda-and-fe-algerian-forest-fires-dataset/Algerian_forest_fires_cleaned_dataset.csv
Shape: (243, 17) | Fire: 137 | Not fire: 106
Regions: {'Bejaia': 243}


In [11]:

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          stratify=y, random_state=42)
model = RandomForestClassifier(n_estimators=400, random_state=42).fit(X_tr, y_tr)
base_f1 = f1_score(y_te, model.predict(X_te))

rng = np.random.default_rng(42)
levels = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]   # Gaussian noise as fraction of std
rows = []
for lvl in levels:
    Xc = X_te.copy()
    for c in Xc.columns:
        Xc[c] = Xc[c] + rng.normal(0, lvl * X_tr[c].std(), size=len(Xc))
    rows.append({"Noise_Level": lvl,
                 "F1": f1_score(y_te, model.predict(Xc)),
                 "F1_Drop": base_f1 - f1_score(y_te, model.predict(Xc))})
robust = pd.DataFrame(rows).round(3)

# per-feature corruption impact at fixed noise = 0.4
feat_rows = []
for c in X_te.columns:
    Xc = X_te.copy()
    Xc[c] = Xc[c] + rng.normal(0, 0.4 * X_tr[c].std(), size=len(Xc))
    feat_rows.append({"Feature": c,
                      "F1_when_corrupted": f1_score(y_te, model.predict(Xc)),
                      "F1_Drop": base_f1 - f1_score(y_te, model.predict(Xc))})
feat_impact = pd.DataFrame(feat_rows).sort_values("F1_Drop", ascending=False).round(3)

out = robust.copy(); out["Baseline_F1"] = round(base_f1, 3)
out.to_csv(f"{OUTDIR}/RQ4_robustness.csv", index=False)
feat_impact.to_csv(f"{OUTDIR}/RQ4_feature_corruption.csv", index=False)
print("Baseline F1:", round(base_f1, 3)); print(robust.to_string(index=False))
print(); print(feat_impact.to_string(index=False))


Baseline F1: 0.988
 Noise_Level    F1  F1_Drop
         0.0 0.988    0.000
         0.1 0.938    0.050
         0.2 0.952    0.036
         0.3 0.965    0.023
         0.4 0.941    0.047
         0.5 0.871    0.117

    Feature  F1_when_corrupted  F1_Drop
        ISI              0.963    0.025
       FFMC              0.964    0.024
         RH              0.988    0.000
Temperature              0.988    0.000
       Rain              0.988    0.000
         Ws              0.988    0.000
        DMC              0.988    0.000
         DC              0.988    0.000
        BUI              0.988    0.000
        FWI              1.000   -0.012


In [12]:

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(robust["Noise_Level"], robust["F1"], "o-", color="#0072B2", lw=2)
axes[0].axhline(base_f1, ls="--", color="gray", label=f"Baseline F1={base_f1:.3f}")
axes[0].set_xlabel("Gaussian Noise Level (fraction of std)")
axes[0].set_ylabel("F1 Score"); axes[0].set_title("Performance vs Noise")
axes[0].legend(frameon=True)
top = feat_impact.head(8)
axes[1].barh(top["Feature"][::-1], top["F1_Drop"][::-1], color="#CC79A7")
axes[1].set_xlabel("F1 Drop when feature corrupted")
axes[1].set_title("Per-Feature Sensitivity (noise=0.4)")
fig.suptitle("RQ4: Robustness to Noisy Sensor Data", y=1.02, fontsize=13)
fig.savefig(f"{OUTDIR}/RQ4_robustness.pdf", bbox_inches="tight")
print("Saved RQ4_robustness.pdf")


Saved RQ4_robustness.pdf
